In [1]:
import pandas as pd
import numpy as np

In [4]:
user_chunks = []

for chunk in pd.read_json(
    "../data/raw/yelp_academic_dataset_user.json",
    lines=True,
    chunksize=50000
):
    chunk = chunk[[
        "user_id",
        "review_count",
        "average_stars",
        "yelping_since"
    ]]
    
    user_chunks.append(chunk)

users = pd.concat(user_chunks, ignore_index=True)

users.head()

,user_id,review_count,average_stars,yelping_since
0,qVc8ODYU5SZjKXVBgXdI7w,585,3.91,2007-01-25 16:47:26
1,j14WgRoU_-2ZE1aw1dXrJg,4333,3.74,2009-01-25 04:35:42
2,2WnXYQFK0hXEoTxPtV2zvg,665,3.32,2008-07-25 10:41:00
3,SZDeASXq7o05mMNLshsdIA,224,4.27,2005-11-29 04:38:33
4,hA5lMy-EnncsH4JoR-hFGQ,79,3.54,2007-01-05 19:40:59


In [5]:
users.shape

(1987897, 4)

In [6]:
business = pd.read_json("../data/raw/yelp_academic_dataset_business.json", lines=True)
business.head()

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."


In [7]:
business.columns

Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'attributes', 'categories', 'hours'],
      dtype='object')

In [8]:
business = business[[
    "business_id",
    "name",
    "stars",
    "review_count",
    "categories"
]]

business.head()

,business_id,name,stars,review_count,categories
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ",5.0,7,"Doctors, Traditional Chinese Medicine, Naturop..."
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,3.0,15,"Shipping Centers, Local Services, Notaries, Ma..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,3.5,22,"Department Stores, Shopping, Fashion, Home & G..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,4.0,80,"Restaurants, Food, Bubble Tea, Coffee & Tea, B..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,4.5,13,"Brewpubs, Breweries, Food"


In [9]:
business.shape

(150346, 5)

In [10]:
user_review_count = users.set_index("user_id")["review_count"].to_dict()

In [11]:
user_avg_stars = users.set_index("user_id")["average_stars"].to_dict()

In [12]:
business_category = business.set_index("business_id")["categories"].to_dict()

In [13]:
business_avg_stars = business.set_index("business_id")["stars"].to_dict()

In [14]:
len(user_review_count)

1987897

In [15]:
len(business_category)

150346

In [16]:
reviews = pd.read_json(
    "../data/raw/yelp_academic_dataset_review.json",
    lines=True,
    nrows=50000
)

reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15


In [17]:
reviews.shape

(50000, 9)

In [18]:
reviews["user_review_count"] = reviews["user_id"].map(user_review_count)

reviews["user_avg_stars"] = reviews["user_id"].map(user_avg_stars)

In [19]:
reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,user_review_count,user_avg_stars
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,33,4.06
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18,10,4.30
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30,1332,4.69
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,9,4.78
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15,126,2.97


In [20]:
reviews["business_category"] = reviews["business_id"].map(business_category)

reviews["business_avg_stars"] = reviews["business_id"].map(business_avg_stars)

In [21]:
reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,user_review_count,user_avg_stars,business_category,business_avg_stars
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,33,4.06,"Restaurants, Breakfast & Brunch, Food, Juice B...",3.0
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18,10,4.30,"Active Life, Cycling Classes, Trainers, Gyms, ...",5.0
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30,1332,4.69,"Restaurants, Breakfast & Brunch",3.5
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,9,4.78,"Halal, Pakistani, Restaurants, Indian",4.0
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15,126,2.97,"Sandwiches, Beer, Wine & Spirits, Bars, Food, ...",4.0


In [22]:
reviews = reviews[[
    "text",
    "stars",
    "user_review_count",
    "user_avg_stars",
    "business_category",
    "business_avg_stars",
    "date"
]]

In [23]:
reviews.head()

,text,stars,user_review_count,user_avg_stars,business_category,business_avg_stars,date
0,"If you decide to eat here, just be aware it is...",3,33,4.06,"Restaurants, Breakfast & Brunch, Food, Juice B...",3.0,2018-07-07 22:09:11
1,I've taken a lot of spin classes over the year...,5,10,4.30,"Active Life, Cycling Classes, Trainers, Gyms, ...",5.0,2012-01-03 15:28:18
2,Family diner. Had the buffet. Eclectic assortm...,3,1332,4.69,"Restaurants, Breakfast & Brunch",3.5,2014-02-05 20:30:30
3,"Wow! Yummy, different, delicious. Our favo...",5,9,4.78,"Halal, Pakistani, Restaurants, Indian",4.0,2015-01-04 00:01:03
4,Cute interior and owner (?) gave us tour of up...,4,126,2.97,"Sandwiches, Beer, Wine & Spirits, Bars, Food, ...",4.0,2017-01-14 20:54:15


In [24]:
reviews.to_csv("../data/processed/reviews_sample.csv", index=False)